In [ ]:
# Instalar primero las dependencias principales sin forzar requests
!pip -q install -U fastapi uvicorn pyngrok langchain langchain-openrouter \
    langchain-huggingface langchain-text-splitters langchain-community \
    sentence-transformers faiss-cpu python-multipart

In [ ]:
# Luego forzar la versión correcta de requests
!pip -q install requests==2.32.4 --upgrade --force-reinstall --no-deps

In [ ]:
# 2. Importar librerías y configurar
import os

from fastapi import FastAPI, UploadFile, File, HTTPException
from pydantic import BaseModel
from google.colab import userdata
from langchain_openrouter import ChatOpenRouter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

In [ ]:
# Configurar OpenRouter
openrouter_api_key = userdata.get("OPENROUTER_API_KEY")
if not openrouter_api_key:
    raise ValueError("Agrega OPENROUTER_API_KEY en los Secrets de Colab.")
os.environ["OPENROUTER_API_KEY"] = openrouter_api_key

In [ ]:
# Crear el modelo de generación (el mismo de siempre)
llm = ChatOpenRouter(
    model="google/gemini-2.5-flash-lite",
    temperature=0.2,
)

In [ ]:
# 3. Crear la aplicación FastAPI
app = FastAPI(
    title="RAG sobre documentos",
    description="API para subir archivos TXT y hacer preguntas sobre su contenido.",
    version="1.0.0",
)

In [ ]:
# Variables globales (las llenaremos en los endpoints)
vector_store = None  # Aquí guardaremos el índice FAISS
file_name = None     # Para recordar el nombre del archivo cargado

print("✅ Entorno listo. Ahora configuraremos los embeddings y endpoints.")

Aquí hacemos una pausa editorial importante.

En los Capítulo 1 y 2 usamos OpenRouter para todo. Para los embeddings, OpenRouter también ofrece opciones, pero son de pago (aunque baratas). Para mantener este ebook **100% gratuito para el lector**, vamos a usar una alternativa que se ejecuta directamente en Colab sin necesidad de API key: **HuggingFace Embeddings**.

Usaremos el modelo `paraphrase-multilingual-MiniLM-L12-v2`. Es rápido, funciona muy bien en español y no pesa demasiado (se descarga en unos segundos dentro de Colab).

**¿Qué hace exactamente un embedding?**  
Imagina que quieres saber si dos frases hablan de lo mismo. Si conviertes ambas frases en coordenadas (vectores) en un espacio de 384 dimensiones, las frases parecidas estarán cerca en ese espacio. FAISS es justamente un buscador de vecinos cercanos en ese espacio.

Agrega esta celda para cargar el modelo de embeddings:



In [ ]:
# 4. Configurar el modelo de embeddings (gratuito, corre localmente en Colab)
print("Cargando modelo de embeddings...")
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)
print("✅ Modelo de embeddings cargado.")

Definir los modelos Pydantic para este proyecto

In [ ]:
# 5. Modelos Pydantic
class AskRequest(BaseModel):
    query: str

class AskResponse(BaseModel):
    query: str
    answer: str
    # Opcional: podemos devolver las fuentes para depurar (lo añadiremos luego)
    # sources: list[str] = []

class UploadResponse(BaseModel):
    filename: str
    chunk_count: int
    status: str

Implementar el endpoint `POST /upload`

In [ ]:
# 6. Endpoint POST /upload
@app.post("/upload", response_model=UploadResponse)
async def upload_file(file: UploadFile = File(...)):
    global vector_store, file_name

    try:
        # Validar que sea un TXT
        if not file.filename.lower().endswith(".txt"):
            raise HTTPException(
                status_code=400,
                detail="Solo se permiten archivos .txt"
            )

        # Leer el contenido del archivo
        content = await file.read()

        try:
            text = content.decode("utf-8")
        except UnicodeDecodeError:
            raise HTTPException(
                status_code=400,
                detail="El archivo debe estar codificado en UTF-8."
            )

        if not text.strip():
            raise HTTPException(
                status_code=400,
                detail="El archivo está vacío."
            )

        # 1. Dividir el texto en fragmentos
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=500,
            chunk_overlap=50,
            length_function=len,
            separators=["\n\n", "\n", " ", ""]
        )

        chunks = splitter.split_text(text)

        if not chunks:
            raise HTTPException(
                status_code=400,
                detail="El texto no pudo dividirse en fragmentos."
            )

        # 2. Crear el índice FAISS
        vector_store = FAISS.from_texts(
            chunks,
            embedding_model
        )

        file_name = file.filename

        return UploadResponse(
            filename=file.filename,
            chunk_count=len(chunks),
            status=f"Documento procesado exitosamente. {len(chunks)} fragmentos creados."
        )

    except HTTPException:
        raise

    except Exception as e:
        raise HTTPException(
            status_code=500,
            detail=f"Error al procesar el archivo: {str(e)}"
        )

Implementar el endpoint `POST /ask`

In [ ]:
# 7. Endpoint POST /ask
@app.post("/ask", response_model=AskResponse)
def ask(request: AskRequest):
    global vector_store

    if vector_store is None:
        raise HTTPException(
            status_code=400,
            detail="Primero debes subir un documento con /upload"
        )

    try:
        # 1. Buscar los fragmentos más relevantes
        docs = vector_store.similarity_search(
            request.query,
            k=4
        )

        context = "\n\n".join(
            doc.page_content for doc in docs
        )

        # 2. Construir el prompt
        prompt = f"""
Eres un asistente que responde preguntas basándose únicamente en el contexto proporcionado.

Si el contexto no contiene información suficiente para responder, responde exactamente:
"No tengo esa información en el documento."

No inventes información que no esté respaldada por el contexto.

Contexto del documento:
---
{context}
---

Pregunta del usuario: {request.query}

Respuesta:
"""

        # 3. Llamar al modelo
        response = llm.invoke(prompt)

        return AskResponse(
            query=request.query,
            answer=response.content
        )

    except HTTPException:
        raise

    except Exception as e:
        raise HTTPException(
            status_code=500,
            detail=f"Error al procesar la pregunta: {str(e)}"
        )

Levantar la API y exponerla con ngrok

In [ ]:
# 8. Levantar servidor
import threading
import time
import uvicorn

PORT = 8000

def run_api():
    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="info")

server_thread = threading.Thread(target=run_api, daemon=True)
server_thread.start()
time.sleep(2)
print(f"✅ Servidor iniciado en el puerto {PORT}")

In [ ]:
# 9. Abrir túnel con ngrok
from pyngrok import ngrok

ngrok_authtoken = userdata.get("NGROK_AUTHTOKEN")
if not ngrok_authtoken:
    raise ValueError("Agrega NGROK_AUTHTOKEN en los Secrets de Colab.")

ngrok.set_auth_token(ngrok_authtoken)
ngrok.kill()
tunnel = ngrok.connect(PORT, "http")
public_url = tunnel.public_url

print("🌍 URL pública temporal:")
print(public_url)
print("\n📚 Documentación interactiva (Swagger UI):")
print(public_url + "/docs")